# 検索した文書を LLM に渡して答えさせる（最小の RAG）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_rag/rag_minimal.ipynb)

NLP クックブック 24-01 ／ RAG（検索拡張生成）

LLM が知らない情報（社内文書、最新情報）について、関連文書を検索してから答えさせたい。GPU なしで動かしたい。

In [1]:
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。CPU でも 1 回答あたり数秒〜十数秒 (メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def chat(messages, max_new_tokens=100):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [2]:
def retrieve(question, k=2):
    sims = doc_emb @ embedder.encode("query: " + question, normalize_embeddings=True)
    return [(int(i), docs[i]) for i in sims.argsort()[::-1][:k]]

def rag(question):
    hits = retrieve(question)
    context = "\n".join(f"[{i}] {d}" for i, d in hits)
    prompt = (f"以下の資料だけに基づいて、質問に一文で答えてください。資料に答えがなければ「わかりません」と答えてください。\n\n"
              f"# 資料\n{context}\n\n# 質問\n{question}")
    return chat([{"role": "user", "content": prompt}]), [i for i, _ in hits]

for q in ["カレーはいくらですか？", "日曜日は何時まで開いていますか？", "クレジットカードで払えますか？"]:
    answer, sources = rag(q)
    print(f"Q: {q}\n  検索なし: {chat([{'role': 'user', 'content': 'クックブック食堂について質問です。' + q}])[:70]}\n  RAG: {answer} (参照: {sources})")

Q: カレーはいくらですか？
  検索なし: クックブック食堂のカレーの価格については、具体的な情報が提供されていないため、正確な金額を伝えることができません。通常、このような質問に対し
  RAG: カレーの通常価格は 950 円です。 (参照: [1, 5])
Q: 日曜日は何時まで開いていますか？
  検索なし: クックブック食堂の営業時間は曜日によって異なる場合がありますが、一般的な営業時間としては、平日は午前11時から午後10時まで、土曜日は午前1
  RAG: 日曜日の営業時間は 20 時までです。 (参照: [0, 3])
Q: クレジットカードで払えますか？
  検索なし: クックブック食堂では、クレジットカードでのお支払いが可能かどうかについては、店舗によって異なる場合があります。そのため、具体的な店舗の情報を
  RAG: いいえ、クレジットカードで払うことはできません。資料の [5] では支払いは現金と QR コード決済に対応しているが、クレジットカードは使えないと明記されています。 (参照: [5, 4])


- RAG = 検索 (Retrieval) + 生成 (Generation)。検索なしだと、モデルは知らない店についてもっともらしい嘘（ハルシネーション）を答えるか、答えられない。
- 精度の大半は検索で決まる。まず「正解の文書が上位 k 件に入っているか」を評価し、ハイブリッド検索やリランキング（検索の章）で改善する。
- もっと軽くしたいなら `sarashina2.2-1b-instruct-v0.1` や `0.5b` に替えられるが、資料を読み違えることが増える。GPU があるなら 7B 以上のモデルにすると安定する。
- トークナイザーを読み込んだあとの数行は、transformers 5 系で sarashina2.2 の分割が学習時と変わってしまう不具合への対処（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照）。この章のほかのレシピも同じ。